In [2]:
import os
import json
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv
# LangChain
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import (
    TextLoader,
    DirectoryLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_tavily import TavilySearch
# LangGraph
from langgraph.graph import (
    StateGraph,
    START,
    END,
)

In [3]:
MODEL_NAME = "openai/gpt-oss-20b"
llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0.6,
    reasoning_format="hidden"
)

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

c:\MY10PMAGENTIC_AI\my10pmenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\MY10PMAGENTIC_AI\my10pmenv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\krish\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode,

In [5]:
embedding_vector = embeddings.embed_query("Hello")
print("Embedding Dimension:",len(embedding_vector))

Embedding Dimension: 384


In [9]:
Data_Path = "C:\DATA"

In [10]:
loader = DirectoryLoader(
    Data_Path,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={
        "encoding": "utf-8"
    }
)

In [12]:
docs = loader.load()
print(
    "Total documents loaded:",
    len(docs)
)

Total documents loaded: 1


In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

In [14]:
new_docs = text_splitter.split_documents(
    docs
)


print(
    "Total chunks:",
    len(new_docs)
)

Total chunks: 14


In [15]:
vector_store = Chroma(
    collection_name="usa_economy",
    embedding_function=embeddings
)

In [16]:
vector_store.add_documents(
    documents=new_docs
)

['bb1b9568-75fa-4b35-a951-e4b66de8b348',
 'eeb7beb5-2fc6-4523-aaff-9ec6416f2b60',
 'c2bbb68a-663a-4acb-a1f0-5bf577b1ebcf',
 '6ca0313e-af85-452e-9188-7de6704e19b4',
 '9baae33e-abb0-465b-b33a-1781f2875909',
 '924d19a9-a3fb-4ac7-a6b9-be4ebd9bcaf2',
 '71918fa2-2be2-47ea-b4ef-e465695a1d80',
 '6efe7065-d361-47af-ad65-7d2ebdb4d404',
 '7fd8ef89-7187-47b0-b590-75625c61fcd9',
 'a8b0fbcc-c143-4a83-b986-f771409b8688',
 '07644749-b559-46cb-ae8b-7abe0a0b723b',
 'd16b16d6-4726-4f4a-9af9-03b923bb5573',
 'd5392ef8-e96f-402d-8b59-659558daed00',
 '5e3dcab0-6391-4aca-b35b-8aa631e0d472']

In [17]:
retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 3
    }
)

In [18]:
test_documents = retriever.invoke(
    "What is GDP of USA?"
)

for document in test_documents:

    print(
        document.page_content
    )

    print("-" * 50)

The United States economy is one of the largest and most diversified economies in the world. It is
--------------------------------------------------
housing costs, and competition from other global economies. The Federal Reserve manages monetary
--------------------------------------------------
The U.S. dollar is widely used in international trade and is also a major global reserve currency.
--------------------------------------------------


In [23]:
from langchain_tavily import TavilySearch

load_dotenv()

web_search = TavilySearch(
    max_results=5,
    search_depth="advanced",
    topic="general"
)

In [24]:
result = web_search.invoke({"query": "What is the latest AI news?"})
print(result)

{'query': 'What is the latest AI news?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://kanerika.com/blogs/recent-developments-in-ai', 'title': 'Recent Developments in AI: What Shipped by November 2026', 'content': 'Announcements such as Gemini 4 Carbon can wait for a price. The recent developments in AI that deserve action this quarter are therefore routing, licenses and the December 2026 dates.\n\nIn short, what shipped by November 2026 rewards teams that test cheaper tiers first and plan around fixed deadlines.\n\n## Frequently Asked Questions\n\n### What are the most recent developments in AI?\n\nAs of November 2026, the biggest changes are tiered model families, open-weight releases and a dated EU AI Act calendar. Anthropic shipped Claude Opus 5.5 on September 22, OpenAI released GPT-6.1 Sol on September 29, and Google published the Gemini 4 Argon model card on October 13. AI agents also moved from pilots into production workflows.\n\n### W

In [25]:
class RouteDecision(BaseModel):
    route: Literal["RAG","LLM","WEB"] = Field(description="Node that should handle the user query")
    reasoning: str = Field(description="Reason for selecting this node")

In [26]:
RouteDecision.model_json_schema()

{'properties': {'route': {'description': 'Node that should handle the user query',
   'enum': ['RAG', 'LLM', 'WEB'],
   'title': 'Route',
   'type': 'string'},
  'reasoning': {'description': 'Reason for selecting this node',
   'title': 'Reasoning',
   'type': 'string'}},
 'required': ['route', 'reasoning'],
 'title': 'RouteDecision',
 'type': 'object'}

In [27]:
llm.invoke("tell me about india?")

AIMessage(content='India is a vast, diverse, and historically rich country located in South Asia. Here’s a broad overview covering its geography, history, culture, politics, and economy:\n\n## 1. Geography & Demographics\n| Feature | Details |\n|---------|---------|\n| **Area** | ~3.29\u202fmillion km² (largest 13th‑ranked country). |\n| **Population** | ~1.4\u202fbillion (2024 estimate), making it the second‑most populous country after China. |\n| **Major Regions** | North (Himalayas, Indo‑Gangetic plain), South (Peninsular India), East (Bengal, Odisha), West (Rajasthan, Gujarat), Central (Chhattisgarh, Madhya Pradesh). |\n| **Major Rivers** | Ganges, Brahmaputra, Yamuna, Godavari, Krishna, Narmada, Tapti. |\n| **Climate** | Tropical in the south, temperate in the north; monsoon season (June‑September) brings most rainfall. |\n\n## 2. Historical Overview\n| Era | Highlights |\n|-----|------------|\n| **Ancient** | Indus Valley Civilization (c. 3300–1300\u202fBCE), Vedic period. |\n| *

In [28]:
supervisor_model = llm.with_structured_output(schema=RouteDecision.model_json_schema(),method="json_schema")

In [29]:
supervisor_model.invoke("tell me about india?")

{'route': 'LLM',
 'reasoning': 'User requests general information about India; LLM can provide comprehensive answer.'}

In [30]:
supervisor_model.invoke("What is GDP of USA?")

{'route': 'LLM',
 'reasoning': 'The question asks for the GDP of the USA, which is a factual query that can be answered by the model with up-to-date information. No additional web search is required.'}

In [31]:
supervisor_model.invoke("can you tell me AI news today?")

{'route': 'WEB',
 'reasoning': 'User requests current AI news, which requires up-to-date information that is best retrieved via a web search.'}

In [32]:
supervisor_model.invoke("can you tell me about today the USA current GDP?")

{'route': 'WEB',
 'reasoning': "The user requests the current GDP of the USA, which requires up-to-date economic data that is likely not available in the LLM's static knowledge base. To provide accurate and timely information, the request should be routed to a web search."}

In [33]:
class ValidationResult(BaseModel):
    passed: bool = Field(description="True if the generated answer passes validation")
    score: int = Field(ge=0,le=10,description="Quality score from 0 to 10")
    feedback: str = Field(description="Reason for validation result")

In [34]:
validator_model = llm.with_structured_output(schema=ValidationResult.model_json_schema(),method="json_schema")
validator_model

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.6', 'langchain': '1.4.9'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000017C53E7A110>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000017C53F40A50>, model_name='openai/gpt-oss-20b', temperature=0.6, reasoning_format='hidden', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'response_format': {'type': 'json_schema', 'json_schema': {'name': 'ValidationResult', 'schema': {'t

In [35]:
class AgentState(TypedDict, total=False):
    # Original user query
    question: str
    # Supervisor decision
    route: str
    # Supervisor reasoning
    route_reasoning: str
    # Context from RAG / Web
    context: str
    # Answer generated by RAG / LLM / Web
    draft_answer: str
    # Validation result
    validation_passed: bool
    validation_score: int
    validation_feedback: str
    # Retry information
    attempts: int
    # Keep track of previously used routes
    used_routes: list[str]
    # Answer shown to user only after validation
    final_answer: str

In [36]:
def supervisor_node(state: AgentState):
    print("\n==============================")
    print("SUPERVISOR NODE")
    print("==============================")
    question = state["question"]
    previous_feedback = state.get("validation_feedback","")
    used_routes = state.get("used_routes",[])
    attempts = state.get("attempts",0)
    supervisor_prompt = f"""
You are the supervisor of an AI system.

Your responsibility is to decide which node should answer
the user's question.

You have three possible nodes:

1. RAG

Use RAG when:
- the question is related to the USA economy
- GDP of USA
- industrial growth of USA
- US economic sectors
- US debt
- US economic strengths
- USA economic information contained in our internal documents

The RAG knowledge base mainly contains information
about the United States economy.


2. WEB

Use WEB when:
- the user asks for current information
- today
- latest
- realtime
- recent news
- current stock priceent weather
- latest economi
- currc information
- information that may have changed recently
- internal RAG documents may be outdated


3. LLM

Use LLM when:
- the question is general knowledge
- casual conversation
- greeting
- conceptual question
- question unrelated to the USA knowledge base
- information does not require current internet data


User Question:
{question}


Routes already attempted:

{used_routes}


Previous validation feedback:

{previous_feedback}


Important:

If a previous answer failed validation,
use the validation feedback to decide whether another
information source should be selected.

Return the best route.
"""


    decision = supervisor_model.invoke(supervisor_prompt)
    route = decision["route"]
    reasoning = decision["reasoning"]

    print("Selected Route:",route)
    print("Reasoning:",reasoning)
    updated_routes = (used_routes + [route])


    return {

        "route": route,

        "route_reasoning": reasoning,

        "attempts": attempts + 1,

        "used_routes": updated_routes,

        # Clear previous attempt
        "context": "",

        "draft_answer": "",

    }

In [37]:
def router(state: AgentState):

    print(
        "\n-> ROUTER ->"
    )


    route = state["route"]


    print(
        "Routing to:",
        route
    )


    return route


In [38]:
def format_docs(documents):
    return "\n\n".join(document.page_content for document in documents)

In [39]:
def rag_node(state: AgentState):
    print("\n==============================")
    print("RAG NODE")
    print("==============================")
    question = state["question"]
    # -----------------------------------
    # Retrieve documents
    # -----------------------------------
    retrieved_docs = retriever.invoke(question)
    context = format_docs(retrieved_docs)
    print("Retrieved Documents:",len(retrieved_docs))

    # -----------------------------------
    # Generate answer
    # -----------------------------------

    rag_prompt = f"""
You are a question-answering assistant.

Answer the user's question using ONLY the supplied
retrieved context.

If the context does not contain enough information,
clearly say that the available internal knowledge
is insufficient.

Do not invent facts.

User Question:

{question}


Retrieved Context:

{context}


Generate a clear and concise answer.
"""
    response = llm.invoke(rag_prompt)
    
    answer = response.content

    print("RAG Answer:", answer)


    return {

        "context": context,

        "draft_answer": answer,

    }

In [40]:
def llm_node(state: AgentState):
    print("\n==============================")
    print("LLM NODE")
    print("==============================")
    question = state["question"]
    prompt = f"""
Answer the following question using your general
knowledge.

Give a clear, accurate and useful response.

If the question requires current or real-time information
that you cannot reliably know, explicitly mention that
real-time information should be checked.

Question:

{question}
"""
    response = llm.invoke(prompt)
    answer = response.content
    print("LLM Answer:",answer)
    
    return {

        "context": "",

        "draft_answer": answer,

    }


In [41]:
def web_node(state: AgentState):
    print(
        "\n=============================="
    )

    print(
        "WEB SEARCH NODE"
    )

    print(
        "=============================="
    )


    question = state["question"]


    # -----------------------------------
    # Search realtime internet
    # -----------------------------------

    search_result = web_search.invoke(

        {
            "query": question
        }

    )


    # -----------------------------------
    # Convert returned data to text
    # -----------------------------------

    if isinstance(
        search_result,
        dict
    ):

        results = search_result.get(
            "results",
            []
        )


        if results:

            web_context_parts = []


            for result in results:

                title = result.get(
                    "title",
                    ""
                )

                url = result.get(
                    "url",
                    ""
                )

                content = result.get(
                    "content",
                    ""
                )


                web_context_parts.append(

                    f"""
Title:
{title}

URL:
{url}

Content:
{content}
"""

                )


            web_context = "\n\n".join(
                web_context_parts
            )


        else:

            web_context = json.dumps(

                search_result,

                indent=2,

                ensure_ascii=False,

                default=str

            )


    else:

        web_context = str(
            search_result
        )


    print(
        "Web search completed."
    )


    # -----------------------------------
    # Generate answer using web context
    # -----------------------------------

    web_prompt = f"""
You are answering a user question using current
web-search results.

Use the web information below to answer the question.

Do not fabricate facts.

When useful, mention the source names or URLs
available in the search result.

User Question:

{question}


Web Search Results:

{web_context}


Generate the best answer based on the current
web information.
"""


    response = llm.invoke(
        web_prompt
    )


    answer = response.content


    print(
        "WEB Answer:",
        answer
    )


    return {

        "context": web_context,

        "draft_answer": answer,

    }

In [42]:
def validator_node(
    state: AgentState
):

    print(
        "\n=============================="
    )

    print(
        "VALIDATOR NODE"
    )

    print(
        "=============================="
    )


    question = state["question"]

    route = state["route"]

    answer = state["draft_answer"]

    context = state.get(
        "context",
        ""
    )


    validation_prompt = f"""
You are the quality-control validator of an AI system.

Evaluate whether the generated answer is acceptable.


USER QUESTION:

{question}


NODE USED:

{route}


SOURCE CONTEXT:

{context}


GENERATED ANSWER:

{answer}


Validate the answer using these criteria:


1. RELEVANCE

Does the answer directly answer the user's question?


2. CORRECTNESS

Does the answer appear logically and factually correct
based on the available evidence?


3. COMPLETENESS

Does it provide enough information to answer the question?


4. HALLUCINATION

Does the answer make unsupported claims?


5. SOURCE GROUNDING

If NODE USED = RAG:

The answer must be supported by the retrieved context.


If NODE USED = WEB:

The answer must be supported by the web search context.


If NODE USED = LLM:

The answer should be suitable for general stable knowledge
and must not pretend to know unavailable real-time facts.


SCORING:

0-4  = poor
5-6  = needs improvement
7    = acceptable but weak
8-10 = good


PASS only if the score is 7 or higher and there is no
major factual or grounding problem.


Return:

passed
score
feedback
"""


    result = validator_model.invoke(
        validation_prompt
    )


    passed = result["passed"]

    score = result["score"]

    feedback = result["feedback"]


    print(
        "Validation Passed:",
        passed
    )

    print(
        "Score:",
        score
    )

    print(
        "Feedback:",
        feedback
    )


    return {

        "validation_passed": passed,

        "validation_score": score,

        "validation_feedback": feedback,

    }


In [43]:
def validation_router(
    state: AgentState
):

    print(
        "\n-> VALIDATION ROUTER ->"
    )


    # -----------------------------------
    # Validation passed
    # -----------------------------------

    if state["validation_passed"]:

        print(
            "Validation PASSED"
        )

        return "Final"


    # -----------------------------------
    # Too many attempts
    # -----------------------------------

    if state.get("attempts",0) >= MAX_ATTEMPTS:
        print(
            "Maximum attempts reached"
        )

        return "Failed"


    # -----------------------------------
    # Validation failed -> supervisor
    # -----------------------------------

    print(
        "Validation FAILED"
    )

    print(
        "Returning to Supervisor..."
    )


    return "Supervisor"
